# Hospital Calling AI Agent


In [17]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio
import ollama
import sqlite3 

In [18]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"

openai = OpenAI(api_key=openai_api_key)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(base_url=ollama_url, api_key="ollama")

MODEL = "llama3.1:8b"
DB = "data.db"

In [19]:
system_message = """
You are a helpful AI calling-center assistant called Maya for a healthcare organization.

* Give short, clear, and courteous answers.
* Help with patient information, documents, appointments, and general inquiries.
* Always be accurate and never guess or make up information.
* Protect patient privacy and only use authorized information.
* If you don't know something, say so.
* For medical concerns, advise the patient to speak with a qualified healthcare professional.
"""

In [20]:
def add_sample_doctors():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()

        cursor.execute("""
            CREATE TABLE IF NOT EXISTS doctors (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT UNIQUE,
                specialty TEXT,
                phone TEXT,
                available_days TEXT
            )
        """)

        doctors = [
            ("Dr. John Silva", "Cardiologist", "0712345678", "Monday, Wednesday, Friday"),
            ("Dr. Sarah Perera", "Dermatologist", "0723456789", "Tuesday, Thursday"),
            ("Dr. Michael Fernando", "Neurologist", "0773456789", "Monday, Thursday"),
            ("Dr. Emily Dias", "Pediatrician", "0764567890", "Wednesday, Friday"),
        ]

        cursor.executemany("""
            INSERT OR IGNORE INTO doctors (name, specialty, phone, available_days)
            VALUES (?, ?, ?, ?)
        """, doctors)

        conn.commit()


add_sample_doctors()


# Tools


In [21]:
def get_doctor_details(name):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT name, specialty, phone, available_days
            FROM doctors
            WHERE name LIKE ?
        """, (f"%{name}%",))    

        return cursor.fetchone()

In [22]:
doctor = get_doctor_details("John")
print(doctor)  

('Dr. John Silva', 'Cardiologist', '0712345678', 'Monday, Wednesday, Friday')


In [23]:
def save_patient(patient_id, name, age, phone, document_type, document_number):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()

        cursor.execute("""
            CREATE TABLE IF NOT EXISTS patients (
                patient_id TEXT PRIMARY KEY,
                name TEXT,
                age INTEGER,
                phone TEXT,
                document_type TEXT,
                document_number TEXT
            )
        """)

        cursor.execute("""
            INSERT INTO patients (
                patient_id, name, age, phone, document_type, document_number
            )
            VALUES (?, ?, ?, ?, ?, ?)
            ON CONFLICT (patient_id) DO UPDATE SET
                name = excluded.name,
                age = excluded.age,
                phone = excluded.phone,
                document_type = excluded.document_type,
                document_number = excluded.document_number
        """, (
            patient_id, name, age, phone, document_type, document_number
        ))

        conn.commit()

In [24]:
save_patient("P001", "Alice Smith", 30, "0712345678", "NIC", "199012345V")

In [25]:
def get_patients():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT patient_id, name, age, phone, document_type, document_number
            FROM patients
        """)
        return cursor.fetchall()

In [26]:
patients = get_patients()
print(patients)

[('P001', 'Alice Smith', 30, '0712345678', 'NIC', '199012345V')]


Define Tools

In [27]:
get_doctor_function = {
    "name": "get_doctor_details",
    "description": "Get doctor details such as name, specialty, phone number, and available days.",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {
                "type": "string",
                "description": "The name of the doctor"
            }
        },
        "required": ["name"],
        "additionalProperties": False
    }
}

save_patient_function = {
    "name": "save_patient",
    "description": "Save or update patient information in the database.",
    "parameters": {
        "type": "object",
        "properties": {
            "patient_id": {
                "type": "string",
                "description": "Unique ID of the patient"
            },
            "name": {
                "type": "string",
                "description": "Full name of the patient"
            },
            "age": {
                "type": "integer",
                "description": "Age of the patient"
            },
            "phone": {
                "type": "string",
                "description": "Phone number of the patient"
            },
            "document_type": {
                "type": "string",
                "description": "Type of identification document"
            },
            "document_number": {
                "type": "string",
                "description": "Identification document number"
            }
        },
        "required": [
            "patient_id",
            "name",
            "age",
            "phone",
            "document_type",
            "document_number"
        ],
        "additionalProperties": False
    }
}

In [28]:
tools = [
    {"type": "function", "function": get_doctor_function},
    {"type": "function","function": save_patient_function}
]

In [29]:
tools

[{'type': 'function',
  'function': {'name': 'get_doctor_details',
   'description': 'Get doctor details such as name, specialty, phone number, and available days.',
   'parameters': {'type': 'object',
    'properties': {'name': {'type': 'string',
      'description': 'The name of the doctor'}},
    'required': ['name'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'save_patient',
   'description': 'Save or update patient information in the database.',
   'parameters': {'type': 'object',
    'properties': {'patient_id': {'type': 'string',
      'description': 'Unique ID of the patient'},
     'name': {'type': 'string', 'description': 'Full name of the patient'},
     'age': {'type': 'integer', 'description': 'Age of the patient'},
     'phone': {'type': 'string', 'description': 'Phone number of the patient'},
     'document_type': {'type': 'string',
      'description': 'Type of identification document'},
     'document_number': {'type': 'string',


In [30]:
def chat(message, history):
    chat_history = []

    for h in history:
        chat_history.append({"role": h['role'], "content": h['content']})

    messages = [{"role": "system", "content": system_message}] + chat_history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=MODEL,messages=messages,tools=tools)

    while response.choices[0].finish_reason == 'tool_calls':
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.extend(response)
        response = ollama.chat.completions.create(model=MODEL,messages=messages, tools=tools)

    return response.choices[0].message.content

In [31]:
def handle_tool_call(message):
    responses = []

    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_doctor_details":
            arguments = json.loads(tool_call.function.arguments)
            name = arguments.get("name")
            doctor_details = get_doctor_details(name)
            responses.append({
                "role": "tool",
                "content": str(doctor_details),
                "tool_call_id": tool_call.id
            })

        elif tool_call.function.name == "save_patient":
            arguments = json.loads(tool_call.function.arguments)
            patient_id = arguments.get("patient_id")
            name = arguments.get("name")
            age = arguments.get("age")
            phone = arguments.get("phone")
            document_type = arguments.get("document_type")
            document_number = arguments.get("document_number")

            save_patient(
                patient_id,
                name,
                age,
                phone,
                document_type,
                document_number
            )

            responses.append({
                "role": "tool",
                "content": "Patient information saved successfully.",
                "tool_call_id": tool_call.id
            })

    return responses

In [32]:
gradio.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
